# 03 · 主模型矩阵

通过 C8 的 `run_matrix.py` 运行 ORTHRUS-ano 与 MSTC-PIDS Full。矩阵 runner 会自动跳过已有 completed marker，并在单个 run 失败后继续其余 run；artifact root 指向 Drive 以持久化。默认仅 seed 0，正式五 seed 时显式改为 `[0, 1, 2, 3, 4]`。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"
DATASET = "THEIA_E3"
SEEDS = [0]  # 正式实验改为 [0, 1, 2, 3, 4]
CONFIGS = [
    PROJECT_ROOT / "config/experiments/baseline.yml",
    PROJECT_ROOT / "config/experiments/mstc_full.yml",
]
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
assert DATASET in {"THEIA_E3", "THEIA_E5"}
assert all(path.is_file() for path in CONFIGS)

In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"),
    "--datasets", DATASET,
    "--configs", ",".join(map(str, CONFIGS)),
    "--seeds", ",".join(map(str, SEEDS)),
    "--artifact-root", str(ARTIFACT_ROOT),
]
result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)
print("run_matrix exit code:", result.returncode)
if result.returncode:
    raise subprocess.CalledProcessError(result.returncode, command)
print("矩阵完成；已有 completed run 会在再次执行时自动 skip。")

## 手动恢复单个训练

不要自动猜测 checkpoint。将下格的 `CHECKPOINT` 指向确认过的 structured `checkpoint.pt` 或其目录，并显式启用。`train,test,evaluate` 表示完整训练恢复；它会恢复 optimizer/RNG/epoch，history 由 replay 重建。

In [ ]:
RUN_MANUAL_RESUME = False
RESUME_CONFIG = PROJECT_ROOT / "config/experiments/mstc_full.yml"
CHECKPOINT = Path("/content/drive/MyDrive/mstc_pids/checkpoints/checkpoint.pt")

if RUN_MANUAL_RESUME:
    if not CHECKPOINT.exists():
        raise FileNotFoundError(CHECKPOINT)
    resume_command = [
        sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"),
        "--dataset", DATASET, "--config", str(RESUME_CONFIG), "--seed", str(SEEDS[0]),
        "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate",
        "--checkpoint", str(CHECKPOINT),
    ]
    subprocess.run(resume_command, cwd=PROJECT_ROOT, check=True)
else:
    print("RUN_MANUAL_RESUME=False；未猜测或加载任何 checkpoint。")